In [ ]:
# -*- coding: utf-8 -*-
"""Copy of Untitled70.ipynb

Automatically generated by Colab.

Original file is located at
    https://colab.research.google.com/drive/18aOa0xZ_uG3xotcYRNlnKzBXJMyWT6rS
"""

!pip install -q \
instaloader requests \
pandas numpy matplotlib seaborn plotly \
wordcloud scikit-learn scipy \
gensim umap-learn \
transformers sentence-transformers \
dash dash-bootstrap-components jupyter-dash \
nltk lda pyLDAvis

In [ ]:
import nltk
nltk.download('punkt')
nltk.download('stopwords')

import json, random, time
import numpy as np
from datetime import datetime, timedelta

MAX_POSTS = 500000

LANGUAGES = ['english','hindi','telugu','tamil','kannada','malayalam','bengali','marathi','gujarati','odia']
weights = [0.35,0.2,0.1,0.1,0.07,0.05,0.05,0.04,0.02,0.02]

HASHTAGS_POOL = ["#love","#instagood","#viral","#trending","#explore","#india","#reels","#food","#travel","#life","#vibes","#aesthetic"]
LOCATIONS = ["Hyderabad","Mumbai","Chennai","Bangalore","Delhi","Pune","Kolkata"]

In [ ]:
def generate_caption():
    return f"{random.choice(['amazing','beautiful','epic','chill'])} moment in {random.choice(LOCATIONS)} feels {random.choice(['happy','excited','tired'])}"

def generate_post(i):
    lang = random.choices(LANGUAGES, weights=weights)[0]

    likes = int(np.random.pareto(2) * 1000)
    likes = max(likes, 30)

    comments = int(likes * random.uniform(0.05,0.2))
    shares = int(likes * random.uniform(0.01,0.05))

    return {
        "post_id": str(100000+i),
        "timestamp": (datetime.now() - timedelta(days=random.randint(0,30))).isoformat(),
        "caption": generate_caption(),
        "language": lang,
        "hashtags": random.sample(HASHTAGS_POOL, random.randint(1,5)),
        "likes_count": likes,
        "comments_count": comments,
        "shares_count": shares,
        "engagement": likes + comments,
        "hashtag_count": random.randint(1,5),
        "location": random.choice(LOCATIONS),
        "media_type": random.choice(["IMAGE","VIDEO","CAROUSEL"])
    }

data = [generate_post(i) for i in range(MAX_POSTS)]

with open("instagram_data.json","w") as f:
    json.dump({"posts":data},f)

print("✅ 500K Dataset Generated")

## Data Loading & Initial Exploration

In [ ]:
import pandas as pd
import json

with open("instagram_data.json") as f:
    raw = json.load(f)

df = pd.DataFrame(raw["posts"])

df['timestamp'] = pd.to_datetime(df['timestamp'], format='mixed')
df['date'] = df['timestamp'].dt.date
df['hour'] = df['timestamp'].dt.hour
df['day_name'] = df['timestamp'].dt.day_name()

df['caption_length'] = df['caption'].str.len()
df['word_count'] = df['caption'].str.split().str.len()

print("✅ Data Loaded Successfully!")

## Dataset Schema & Quality Report

In [ ]:
# =========================
# PRINT SCHEMA
# =========================
print("="*60)
print("DATASET SCHEMA")
print("="*60)

print(f"Total posts : {len(df):,}")
print(f"Total columns : {len(df.columns)}")
print(f"Languages covered : {df['language'].nunique()}")

print("\nData Types:")
print(df.dtypes)

print("\nBasic Statistics:")
print(df.describe())

print("\nTop Locations:")
print(df['location'].value_counts().head())

In [ ]:
# =========================
# DATA QUALITY REPORT
# =========================
print("\n" + "="*60)
print("DATA QUALITY REPORT")
print("="*60)

missing_values = df.isnull().sum()
print(f"\nMissing Values:\n{missing_values[missing_values > 0] if missing_values.sum() > 0 else 'None'}")

duplicate_posts = df.duplicated(subset=['post_id']).sum()
print(f"\nDuplicate Posts: {duplicate_posts}")

outliers = {
    'likes_outliers': len(df[df['likes_count'] > df['likes_count'].quantile(0.95)]),
    'engagement_outliers': len(df[df['engagement'] > df['engagement'].quantile(0.95)])
}
print(f"\nOutliers Detected:\n{outliers}")

## Visualization & Analysis

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

sns.set(style="whitegrid")

# Language distribution
df['language'].value_counts().plot(kind='bar', figsize=(12,5))
plt.title("Language Distribution")
plt.tight_layout()
plt.show()

# Likes distribution
sns.histplot(np.log1p(df['likes_count']), bins=40)
plt.title("Likes Distribution (log)")
plt.show()

# Engagement vs hashtags
sample = df.sample(5000)
plt.scatter(sample['hashtag_count'], sample['engagement'], alpha=0.3)
plt.title("Hashtags vs Engagement")
plt.show()

In [ ]:
from collections import Counter

fig, axes = plt.subplots(1,2, figsize=(14,5))

num_cols = ['likes_count','comments_count','shares_count','engagement','caption_length','word_count','hashtag_count']

sns.heatmap(df[num_cols].corr(), annot=True, ax=axes[0], cmap='coolwarm')
axes[0].set_title("Correlation Matrix")

# Top hashtags
all_tags = []
for tags in df['hashtags']:
    all_tags.extend(tags)

top_tags = Counter(all_tags).most_common(10)
tags_df = pd.DataFrame(top_tags, columns=['tag','count'])

sns.barplot(data=tags_df, y='tag', x='count', ax=axes[1], palette='viridis')
axes[1].set_title("Top Hashtags")

plt.tight_layout()
plt.show()

## Text Processing & Sentiment Analysis

In [ ]:
import re

def clean_text(text):
    text = re.sub(r"http\S+","",text)
    text = re.sub(r"[^\w\s]","",text)
    return text.lower()

df['cleaned_text'] = df['caption'].apply(clean_text)

POS = ['amazing','happy','good','great','awesome']
NEG = ['bad','sad','worst','hate']

def sentiment(text):
    score = 0
    for w in POS:
        if w in text: score += 1
    for w in NEG:
        if w in text: score -= 1
    return "positive" if score>0 else "negative" if score<0 else "neutral"

df['sentiment'] = df['cleaned_text'].apply(sentiment)

fig, axes = plt.subplots(1,2, figsize=(14,5))

df['sentiment'].value_counts().plot(kind='bar', ax=axes[0], color=['green', 'red', 'gray'])
axes[0].set_title("Sentiment Distribution")
axes[0].set_ylabel("Count")

sns.boxplot(data=df, x='sentiment', y='engagement', ax=axes[1])
axes[1].set_title("Engagement by Sentiment")

plt.tight_layout()
plt.show()

In [ ]:
from wordcloud import WordCloud

text = " ".join(df['cleaned_text'].sample(40000))

wc = WordCloud(width=1200,height=700, colormap='viridis').generate(text)

plt.figure(figsize=(15,8))
plt.imshow(wc)
plt.axis("off")
plt.title("Word Cloud - Most Frequent Words in Captions")
plt.tight_layout()
plt.show()

## Advanced NLP: Topic Modeling (LDA)

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.decomposition import LatentDirichletAllocation

sample_texts = df['cleaned_text'].sample(5000).tolist()

vectorizer = CountVectorizer(max_features=500, stop_words='english')
X_lda = vectorizer.fit_transform(sample_texts)

lda = LatentDirichletAllocation(n_components=5, random_state=42)
lda.fit(X_lda)

print("\n" + "="*60)
print("TOPIC MODELING - LDA ANALYSIS")
print("="*60)
print("\nTop 5 Topics Identified:")
for idx, topic in enumerate(lda.components_):
    top_words_idx = topic.argsort()[-5:][::-1]
    top_words = [vectorizer.get_feature_names_out()[i] for i in top_words_idx]
    print(f"Topic {idx+1}: {', '.join(top_words)}")

## Embeddings & Vectorization

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from gensim.models import Word2Vec, FastText
from sentence_transformers import SentenceTransformer

sample_df = df.sample(2000)

corpus = sample_df['cleaned_text'].tolist()
tokens = [t.split() for t in corpus]

# TF-IDF
tfidf = TfidfVectorizer(max_features=2000)
X_tfidf = tfidf.fit_transform(corpus).toarray()

# Word2Vec
w2v = Word2Vec(sentences=tokens, vector_size=100, window=5, min_count=1, workers=4)

def avg_w2v(tokens):
    return np.mean([w2v.wv[t] for t in tokens if t in w2v.wv], axis=0)

X_w2v = np.array([avg_w2v(t) for t in tokens])

# FastText
ft = FastText(sentences=tokens, vector_size=100, workers=4)

def avg_ft(tokens):
    return np.mean([ft.wv[t] for t in tokens if t in ft.wv], axis=0)

X_ft = np.array([avg_ft(t) for t in tokens])

# mBERT
model = SentenceTransformer('paraphrase-multilingual-MiniLM-L12-v2')
X_bert = model.encode(corpus)

print("TFIDF:",X_tfidf.shape)
print("Word2Vec:",X_w2v.shape)
print("FastText:",X_ft.shape)
print("BERT:",X_bert.shape)

## Dimensionality Reduction & Clustering

In [ ]:
from sklearn.manifold import TSNE

X = X_tfidf[:2000]

tsne = TSNE(n_components=2, random_state=42)
X_2d = tsne.fit_transform(X)

plt.figure(figsize=(12,8))
scatter = plt.scatter(X_2d[:,0], X_2d[:,1], c=sample_df['likes_count'], cmap='viridis', alpha=0.6, s=50)
plt.colorbar(scatter, label='Likes Count')
plt.title("t-SNE Visualization of Text Embeddings")
plt.xlabel("t-SNE Dimension 1")
plt.ylabel("t-SNE Dimension 2")
plt.show()

In [ ]:
from sklearn.cluster import KMeans

kmeans = KMeans(n_clusters=5, random_state=42, n_init=10)
labels = kmeans.fit_predict(X_tfidf)

df_sample = df.sample(len(labels)).reset_index(drop=True)
df_sample['cluster'] = labels

fig, axes = plt.subplots(1,2, figsize=(14,5))

sns.scatterplot(x=df_sample['likes_count'], y=df_sample['engagement'], hue=df_sample['cluster'], ax=axes[0], palette='Set2')
axes[0].set_title("Engagement vs Likes (Colored by Cluster)")

sns.boxplot(data=df_sample, x='cluster', y='engagement', ax=axes[1])
axes[1].set_title("Engagement Distribution by Cluster")

plt.tight_layout()
plt.show()

## Time-Series Analysis

In [ ]:
print("\n" + "="*60)
print("TIME-SERIES ANALYSIS")
print("="*60)

df_ts = df.set_index('timestamp').resample('D').agg({
    'likes_count': 'sum',
    'comments_count': 'sum',
    'engagement': 'sum',
    'post_id': 'count'
}).rename(columns={'post_id': 'posts_count'})

fig, axes = plt.subplots(2,2, figsize=(15,10))

axes[0,0].plot(df_ts.index, df_ts['posts_count'], marker='o')
axes[0,0].set_title("Daily Post Count Trend")
axes[0,0].set_ylabel("Number of Posts")

axes[0,1].plot(df_ts.index, df_ts['likes_count'], color='orange', marker='o')
axes[0,1].set_title("Daily Likes Trend")
axes[0,1].set_ylabel("Total Likes")

axes[1,0].plot(df_ts.index, df_ts['engagement'], color='green', marker='o')
axes[1,0].set_title("Daily Engagement Trend")
axes[1,0].set_ylabel("Total Engagement")

axes[1,1].plot(df_ts.index, df_ts['comments_count'], color='red', marker='o')
axes[1,1].set_title("Daily Comments Trend")
axes[1,1].set_ylabel("Total Comments")

plt.tight_layout()
plt.show()

## Machine Learning Models

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, f1_score

X = df[['hashtag_count','caption_length']]
y = df['sentiment']

X_train, X_test, y_train, y_test = train_test_split(X,y,test_size=0.2, random_state=42)

model = LogisticRegression(max_iter=1000)
model.fit(X_train, y_train)

y_pred = model.predict(X_test)

print("\n" + "="*60)
print("LOGISTIC REGRESSION - SENTIMENT CLASSIFICATION")
print("="*60)
print(classification_report(y_test, y_pred))
print(f"F1-Score: {f1_score(y_test, y_pred, average='weighted'):.4f}")

In [ ]:
from sklearn.ensemble import RandomForestClassifier

rf = RandomForestClassifier(n_estimators=100, random_state=42)
rf.fit(X_train, y_train)

y_pred_rf = rf.predict(X_test)

print("\n" + "="*60)
print("RANDOM FOREST - SENTIMENT CLASSIFICATION")
print("="*60)
print(classification_report(y_test, y_pred_rf))
print(f"F1-Score: {f1_score(y_test, y_pred_rf, average='weighted'):.4f}")

feature_importance = pd.DataFrame({
    'feature': X.columns,
    'importance': rf.feature_importances_
}).sort_values('importance', ascending=False)

print("\nFeature Importance:")
print(feature_importance)

In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay, roc_curve, auc

X = df[['likes_count','comments_count','shares_count','hashtag_count']]
y = (df['engagement'] > df['engagement'].median()).astype(int)

X_train, X_test, y_train, y_test = train_test_split(X,y,test_size=0.2, random_state=42)

# Scale features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

model = LogisticRegression(max_iter=1000)
model.fit(X_train_scaled, y_train)

y_pred = model.predict(X_test_scaled)
y_pred_proba = model.predict_proba(X_test_scaled)[:,1]

fig, axes = plt.subplots(1,2, figsize=(14,5))

cm = confusion_matrix(y_test, y_pred)
disp = ConfusionMatrixDisplay(cm)
disp.plot(ax=axes[0])
axes[0].set_title("ML Confusion Matrix")

fpr, tpr, _ = roc_curve(y_test, y_pred_proba)
roc_auc = auc(fpr, tpr)
axes[1].plot(fpr, tpr, label=f'ROC curve (AUC = {roc_auc:.3f})')
axes[1].plot([0, 1], [0, 1], 'k--', label='Random')
axes[1].set_xlabel('False Positive Rate')
axes[1].set_ylabel('True Positive Rate')
axes[1].set_title('ROC Curve')
axes[1].legend()

plt.tight_layout()
plt.show()

## Deep Learning Models

In [ ]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout
import warnings
warnings.filterwarnings('ignore')

model = Sequential([
    Dense(128, activation='relu', input_shape=(4,)),
    Dropout(0.3),
    Dense(64, activation='relu'),
    Dropout(0.3),
    Dense(32, activation='relu'),
    Dense(1, activation='sigmoid')
])

model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

history = model.fit(X_train_scaled, y_train, epochs=10, batch_size=32, validation_data=(X_test_scaled,y_test), verbose=0)

# Plot accuracy and loss
fig, axes = plt.subplots(1,2, figsize=(14,5))

axes[0].plot(history.history['accuracy'], label='train', marker='o')
axes[0].plot(history.history['val_accuracy'], label='val', marker='o')
axes[0].set_title("DL Model Accuracy")
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("Accuracy")
axes[0].legend()
axes[0].grid()

axes[1].plot(history.history['loss'], label='train', marker='o')
axes[1].plot(history.history['val_loss'], label='val', marker='o')
axes[1].set_title("DL Model Loss")
axes[1].set_xlabel("Epoch")
axes[1].set_ylabel("Loss")
axes[1].legend()
axes[1].grid()

plt.tight_layout()
plt.show()

## Performance Metrics & ROI Analysis

In [ ]:
print("\n" + "="*60)
print("ROI & ENGAGEMENT PERFORMANCE METRICS")
print("="*60)

# Calculate engagement rate
df['engagement_rate'] = (df['engagement'] / (df['likes_count'] + 1)) * 100

# Performance by media type
media_performance = df.groupby('media_type').agg({
    'likes_count': 'mean',
    'comments_count': 'mean',
    'engagement': 'mean',
    'engagement_rate': 'mean',
    'post_id': 'count'
}).rename(columns={'post_id': 'total_posts'})

print("\nPerformance by Media Type:")
print(media_performance.round(2))

# Performance by location
location_performance = df.groupby('location').agg({
    'likes_count': 'mean',
    'engagement': 'mean',
    'post_id': 'count'
}).rename(columns={'post_id': 'total_posts'}).sort_values('likes_count', ascending=False)

print("\nPerformance by Location:")
print(location_performance.round(2))

# Performance by language
language_performance = df.groupby('language').agg({
    'likes_count': 'mean',
    'engagement': 'mean',
    'engagement_rate': 'mean',
    'post_id': 'count'
}).rename(columns={'post_id': 'total_posts'}).sort_values('engagement', ascending=False)

print("\nPerformance by Language:")
print(language_performance.round(2))

In [ ]:
# Visualize performance metrics
fig, axes = plt.subplots(2,2, figsize=(15,10))

media_performance['likes_count'].plot(kind='bar', ax=axes[0,0], color='skyblue')
axes[0,0].set_title("Avg Likes by Media Type")
axes[0,0].set_ylabel("Average Likes")

media_performance['engagement'].plot(kind='bar', ax=axes[0,1], color='lightcoral')
axes[0,1].set_title("Avg Engagement by Media Type")
axes[0,1].set_ylabel("Average Engagement")

location_performance['likes_count'].plot(kind='barh', ax=axes[1,0], color='lightgreen')
axes[1,0].set_title("Avg Likes by Location")
axes[1,0].set_xlabel("Average Likes")

language_performance['engagement'].plot(kind='bar', ax=axes[1,1], color='gold')
axes[1,1].set_title("Avg Engagement by Language")
axes[1,1].set_ylabel("Average Engagement")

plt.tight_layout()
plt.show()

## Interactive Visualizations (Plotly)

In [ ]:
import plotly.express as px

df_plot = df.sample(5000)

fig1 = px.scatter(df_plot,
                  x='comments_count',
                  y='engagement',
                  color='sentiment',
                  size='likes_count',
                  hover_data=['language','media_type'],
                  title="Engagement vs Comments (Colored by Sentiment)")

fig2 = px.histogram(df_plot,
                    x='language',
                    color='sentiment',
                    title="Language Distribution by Sentiment",
                    barmode='group')

fig3 = px.box(df_plot,
              x='sentiment',
              y='engagement',
              color='media_type',
              title="Engagement by Sentiment & Media Type")

fig1.show()
fig2.show()
fig3.show()

In [ ]:
df_sample = df.sample(3000)

fig1 = px.scatter(df_sample, x='likes_count', y='engagement', color='language', size='comments_count',
                  title="Likes vs Engagement by Language", hover_data=['location'])

fig2 = px.histogram(df_sample, x='location', color='media_type', 
                    title="Posts by Location and Media Type", barmode='group')

fig3 = px.box(df_sample, x='media_type', y='engagement', color='sentiment',
              title="Engagement Distribution by Media Type & Sentiment")

fig1.show()
fig2.show()
fig3.show()

## Export Data

In [ ]:
# Save the DataFrame to a CSV file
df.to_csv("instagram_preprocessed.csv", index=False)
print("✅ Data exported to instagram_preprocessed.csv")

## Streamlit Dashboard

This section contains the interactive Streamlit dashboard code. Save as `app.py` and run with `streamlit run app.py`

In [ ]:
# ============== STREAMLIT DASHBOARD CODE ==============
# Save this as app.py and run: streamlit run app.py

streamlit_code = '''
import streamlit as st
import pandas as pd
import plotly.express as px
import numpy as np

# PAGE CONFIG
st.set_page_config(page_title="Instagram AI Dashboard", layout="wide")

# LOAD DATA
@st.cache_data
def load_data():
    return pd.read_csv("instagram_preprocessed.csv")

df = load_data()

# Add engagement rate
if "engagement_rate" not in df.columns:
    df["engagement_rate"] = (df["engagement"] / (df["likes_count"] + 1)) * 100

# SIDEBAR
st.sidebar.title("📊 Controls")
language = st.sidebar.selectbox("Language", df["language"].unique())

if "media_type" in df.columns:
    media = st.sidebar.selectbox("Media Type", df["media_type"].unique())
    df_filtered = df[(df["language"] == language) & (df["media_type"] == media)]
else:
    df_filtered = df[df["language"] == language]

page = st.sidebar.radio("Navigate", [
    "Overview",
    "Engagement",
    "Advanced",
    "Performance Metrics",
    "Data"
])

# HEADER
st.title("🚀 Instagram AI Analytics Dashboard")

c1, c2, c3, c4, c5 = st.columns(5)
c1.metric("Total Posts", len(df_filtered))
c2.metric("Avg Likes", f"{int(df_filtered['likes_count'].mean()):,}")
c3.metric("Avg Comments", f"{int(df_filtered['comments_count'].mean())}")
c4.metric("Avg Engagement", f"{int(df_filtered['engagement'].mean()):,}")
c5.metric("Avg Engagement Rate", f"{df_filtered['engagement_rate'].mean():.2f}%")

st.markdown("---")

# OVERVIEW PAGE
if page == "Overview":
    col1, col2 = st.columns(2)
    
    lang_counts = df["language"].value_counts().reset_index()
    lang_counts.columns = ["language", "count"]
    
    fig1 = px.bar(lang_counts, x="language", y="count", color="count",
                  title="🌍 Language Distribution", color_continuous_scale="Viridis")
    col1.plotly_chart(fig1, use_container_width=True)
    
    if "sentiment" in df.columns:
        fig2 = px.pie(df_filtered, names="sentiment", title="😊 Sentiment Distribution")
        col2.plotly_chart(fig2, use_container_width=True)

# ENGAGEMENT PAGE
elif page == "Engagement":
    col1, col2 = st.columns(2)
    
    fig1 = px.scatter(df_filtered.sample(min(3000, len(df_filtered))),
                      x="hashtag_count", y="engagement", color="language",
                      size="likes_count", title="📊 Hashtags vs Engagement")
    col1.plotly_chart(fig1, use_container_width=True)
    
    fig2 = px.histogram(df_filtered, x="likes_count", nbins=50,
                        title="📉 Likes Distribution")
    col2.plotly_chart(fig2, use_container_width=True)
    
    hourly_data = df_filtered.groupby("hour")[["engagement"]].mean().reset_index()
    fig3 = px.line(hourly_data, x="hour", y="engagement", markers=True,
                   title="Engagement by Hour of Day")
    st.plotly_chart(fig3, use_container_width=True)

# ADVANCED PAGE
elif page == "Advanced":
    col1, col2 = st.columns(2)
    
    num_cols = ["likes_count","comments_count","shares_count",
                "engagement","caption_length","word_count","hashtag_count"]
    existing_cols = [c for c in num_cols if c in df_filtered.columns]
    
    if len(existing_cols) > 0:
        corr = df_filtered[existing_cols].corr()
        fig1 = px.imshow(corr, text_auto=True, title="📊 Correlation Matrix")
        col1.plotly_chart(fig1, use_container_width=True)
    
    if "hashtags" in df_filtered.columns:
        tags = df_filtered["hashtags"].astype(str).str.split().explode()
        top_tags = tags.value_counts().head(10).reset_index()
        top_tags.columns = ["tag", "count"]
        fig2 = px.bar(top_tags, x="count", y="tag", orientation="h",
                      title="🔥 Top Hashtags")
        col2.plotly_chart(fig2, use_container_width=True)

# PERFORMANCE METRICS PAGE
elif page == "Performance Metrics":
    st.subheader("📊 Performance Analysis")
    col1, col2 = st.columns(2)
    
    media_perf = df_filtered.groupby("media_type").agg({
        "likes_count": "mean",
        "engagement": "mean"
    }).reset_index()
    fig1 = px.bar(media_perf, x="media_type", y=["likes_count", "engagement"],
                  title="Performance by Media Type", barmode="group")
    col1.plotly_chart(fig1, use_container_width=True)
    
    location_perf = df_filtered.groupby("location")["engagement"].mean().reset_index().sort_values("engagement", ascending=True)
    fig2 = px.bar(location_perf, y="location", x="engagement", orientation="h",
                  title="Avg Engagement by Location", color="engagement")
    col2.plotly_chart(fig2, use_container_width=True)

# DATA PAGE
elif page == "Data":
    st.subheader("📋 Data Preview")
    st.dataframe(df_filtered.head(50))
    st.download_button("Download Data", df_filtered.to_csv(index=False),
                       file_name="filtered_data.csv")

st.markdown("---")
st.markdown("📊 Built with Streamlit | AI & ML Project | Data Quality Assured")
'''

print("✅ Streamlit dashboard code generated!")
print("\n📝 To use the dashboard:")
print("1. Copy the code above into a file called 'app.py'")
print("2. Run: streamlit run app.py")

## Model Comparison Summary

In [ ]:
print("\nModel Comparison Summary:")
print("""
1. Logistic Regression → Fast, baseline, interpretable
2. Random Forest → Better handling non-linearity, feature importance
3. Neural Network → Deep learning performance, complex patterns
4. BERT → Best contextual understanding, NLP-specific
5. Topic Modeling (LDA) → Discover hidden topics in captions
6. Time-Series Analysis → Temporal patterns and trends
""")